# 3 · Fine-tune Prithvi on image chips

In [notebook 1](01_model.ipynb) we trained on four Sentinel-2 bands. In [notebook 2](02_sources.ipynb) we used an observation’s grid and date to read a crop. Prithvi Tiny uses both ideas. For each image chip, we’ll give it the four bands, the chip’s center and the acquisition date.

Reuse the [Dynamic World teaching data](https://huggingface.co/datasets/fatmur/dynamicworld-300) and [human annotations](https://doi.org/10.1594/PANGAEA.933475) downloaded in notebook 0. The first run downloads about 130 MB of encoder weights.

## Load the class definitions

In [1]:
from pathlib import Path

import lightning as L
import torch
import yaml
from huggingface_hub import snapshot_download
from torch import nn
from torch.utils.data import DataLoader, Dataset

import geosave as gs

# One seed for the whole notebook, so the model's first weights and the shuffled
# batches repeat from run to run.
L.seed_everything(0)

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300",
        repo_type="dataset",
        revision="4d3353a45d76c55a92506ef89aedf43efafc3bf6",
        local_dir="data/dynamic_world",
    )
)
schemes = yaml.safe_load((root / "labels.yaml").read_text())

Seed set to 0


Fetching 913 files:   0%|          | 0/913 [00:00<?, ?it/s]

Use the same class lookup and zero-fill as in notebook 1.

In [2]:
import numpy as np
import xarray as xr

target = schemes["dynamic_world"]
class_names = dict(enumerate(target["classes"]))
class_colors = dict(enumerate(target["colors"]))
num_classes = len(class_names)
ignore_index = target["ignore_index"]

remap = target["remap"]
lookup = np.array([remap[code] for code in sorted(remap)], dtype="uint8")

def remap_labels(label: xr.DataArray) -> xr.DataArray:
    return label.copy(data=lookup[label.values])

def fill_missing_pixels(image: xr.Dataset) -> xr.Dataset:
    return image.fillna(0)

## Choose the pretrained encoder

Use the encoder → decoder → head recipe from notebook 1. Prithvi EO 2.0 Tiny TL has pretrained weights for the blue, green and red bands. It learns the input weights for B08, the broad near-infrared band, during fine-tuning.

In [3]:
from geosave.ml import ModelChain, build_model

CHIP_SIZE = 128
model = build_model({
    "encoder": {
        "name": "prithvi_tl",
        "init_args": {
            "model_name": "prithvi_eo_v2_tiny_tl",
            "pretrained": True,
            "bands": ["BLUE", "GREEN", "RED", "NIR_BROAD"],
            "input_size": CHIP_SIZE,
        },
    },
    "decoder": {"name": "unet"},
    "head": {"name": "segmentation", "init_args": {"num_classes": num_classes}},
})

2026-11-19 16:41:06,714 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-tiny-TL/resolve/main/Prithvi_EO_V2_tiny_TL.pt "HTTP/1.1 302 Found"
2026-11-19 16:41:06,988 - INFO - Loaded weights for HLSBands.BLUE in position 0 of patch embed
2026-11-19 16:41:06,994 - INFO - Loaded weights for HLSBands.GREEN in position 1 of patch embed
2026-11-19 16:41:07,007 - INFO - Loaded weights for HLSBands.RED in position 2 of patch embed


## Keep the input and target separate

As in notebook 1, one sample is an image and its target. The task determines what the target means. Here it is a class mask on the image grid. A scene classifier could use one class ID per image. A YOLO detector could use a list of class IDs and bounding boxes.

A DataTree is useful when several raster inputs belong together, such as optical and radar images. Each named node holds a Dataset. Here the four optical bands already share one Dataset, and the mask is used to score predictions.

## Read a random training crop

One training index selects an image and its mask. Each visit to that index draws a fresh 128 × 128 window. Crop the image before loading pixels, then read the mask on the same grid. As in notebook 1, `read_sample` returns rasters you can inspect and `__getitem__` turns them into model tensors.

In [ ]:
BANDS = ["B02", "B03", "B04", "B08"]
CONTEXT = ["centroid", "day_of_year"]


class DynamicWorld(Dataset):
    def __init__(self, images: Sequence[Path], labels: Sequence[Path]):
        self.images = images
        self.labels = labels

    def __len__(self) -> int:
        return len(self.images)

    def read_sample(self, index: int):
        with (
            gs.open_raster(self.images[index], mask_and_scale=True) as image,
            gs.open_raster(self.labels[index]) as annotation,
        ):
            label = annotation["band_1"]

            image = gs.random_crop(image, size=CHIP_SIZE)
            label = label.gs.crop_to_grid(image.gs.geobox)

            image = fill_missing_pixels(image).astype("float32").load()
            label = remap_labels(label)
        return image, label

    def __getitem__(self, index: int):
        image, label = self.read_sample(index)
        pixels, context = image.gs.to_tensor(context=CONTEXT)
        target, _ = label.gs.to_tensor(dtype="int64")
        return {"image": pixels, **context}, target

## Read whole images for validation and test

As in notebook 1, `read_sample(index)` returns the image and its reference mask on the same grid. Here the image stays lazy. `__getitem__` names the image for the predictor and converts the remapped mask to an integer tensor for scoring.

In [ ]:
class DynamicWorldEval(Dataset):
    def __init__(self, images: Sequence[Path], labels: Sequence[Path]) -> None:
        self.images = images
        self.labels = labels

    def __len__(self) -> int:
        return len(self.images)

    def read_sample(self, index: int) -> tuple[xr.Dataset, xr.DataArray]:
        scene = gs.open_raster(self.images[index], mask_and_scale=True)
        image = fill_missing_pixels(scene[BANDS]).astype("float32")
        with gs.open_raster(self.labels[index]) as annotation:
            label = remap_labels(annotation["band_1"])
            
        return image, label

    def __getitem__(self, index: int) -> tuple[xr.DataTree, torch.Tensor]:
        image, label = self.read_sample(index)
        target, _ = label.gs.to_tensor(dtype="int64")
        return image.gs.to_tree("image"), target

## Choose the dataset for each split

Training shuffles batches of two random crops. For validation and test, `scene.parquet` records the matching image and label paths for each annotation. These loaders yield one whole image and mask at a time. With `batch_size=None`, the evaluation loaders pass each sample through as it is. The predictor moves chip batches to the model’s device.

In [ ]:
class DynamicWorldDataModule(L.LightningDataModule):
    def __init__(self, root: Path) -> None:
        super().__init__()
        self.root = root

    def evaluation_dataset(self, split: str) -> DynamicWorldEval:
        scenes = pd.read_parquet(
            self.root / "scene.parquet", columns=["image_path", "label_path"],
            filters=[("split", "=", split)],
        )
        scenes = scenes.sort_values("label_path").head(2)
        images = [self.root / path for path in scenes["image_path"]]
        labels = [self.root / path for path in scenes["label_path"]]
        return DynamicWorldEval(images, labels)

    def setup(self, stage: str) -> None:
        if stage == "fit":
            labels = sorted((self.root / "label/train/expert").glob("*.tif"))[:8]
            images = [self.root / "image/train" / label.name for label in labels]
            self.train = DynamicWorld(images, labels)
        if stage in ("fit", "validate"):
            self.val = self.evaluation_dataset("val")
        if stage == "test":
            self.test = self.evaluation_dataset("test")

    def train_dataloader(self) -> DataLoader:
        return DataLoader(self.train, batch_size=2, shuffle=True)

    def val_dataloader(self) -> DataLoader:
        return DataLoader(self.val, batch_size=None, collate_fn=lambda sample: sample)

    def test_dataloader(self) -> DataLoader:
        return DataLoader(self.test, batch_size=None, collate_fn=lambda sample: sample)

    def transfer_batch_to_device(self, batch, device: torch.device, dataloader_idx: int):
        if self.trainer.training:
            return super().transfer_batch_to_device(batch, device, dataloader_idx)
        return batch  # The predictor reads the lazy image one chip batch at a time.

Call `setup("fit")` here to inspect one batch from each loader. The trainer calls `setup` automatically during fitting and testing.

In [ ]:
data = DynamicWorldDataModule(root)
data.setup("fit")

inputs, masks = next(iter(data.train_dataloader()))
print(inputs["image"].shape, masks.shape)

image, mask = next(iter(data.val_dataloader()))
print(image["image"].sizes, mask.shape)

## Build a whole-scene predictor

Call `predictor(model, sample)` with one prepared image. `RasterChipper` locates the windows and `RasterMerger` uses the same layout to place each chip’s logits back on the image grid. With a limit of 16, a 49-chip scene takes four model calls of 16, 16, 16 and 1. The merged logits for one scene stay in host memory.

Notebook 4 uses this same predictor in a pipeline. `measure_chips()` tells the pipeline the chip size and stride. `@chain_step()` connects the returned `output` to the postprocessor.

In [ ]:
from dataclasses import dataclass
from itertools import batched

from torch import nn
from torch.utils.data import default_collate

from geosave.ml import chain_step
from geosave.ml.processing.raster.prediction.base import RasterPredictor


@dataclass
class LandCoverPredictor(RasterPredictor):
    size: int = 128
    stride: int = 64
    batch_size: int = 16

    def measure_chips(self) -> tuple[int, int]:
        return self.size, self.stride

    @chain_step()
    def __call__(self, model: nn.Module, sample: xr.DataTree) -> xr.DataArray:
        chips = gs.RasterChipper([sample], size=self.size, stride=self.stride)
        merger = gs.RasterMerger(chips.layout, window="hann")
        device = next(model.parameters()).device

        for chip_indices in batched(range(len(chips)), self.batch_size):
            chip_inputs = []
            for index in chip_indices:
                pixels, context = chips[index].gs.to_tensor(context=["centroid", "year", "day_of_year"])
                chip_inputs.append({**pixels, **context})

            batch = default_collate(chip_inputs)
            inputs = {name: tensor.to(device) for name, tensor in batch.items()}
            with torch.inference_mode():
                batch_logits = model(**inputs).float().cpu().numpy()
            for index, chip_logits in zip(chip_indices, batch_logits, strict=True):
                merger.add(index, chip_logits)

        output = merger.merge()
        return output

## Fine-tune on the same split

Training follows notebook 1. Flip the image and mask together, run the model and compute cross-entropy on labeled pixels. A random crop can contain only ignored pixels, so divide the summed loss by at least one.

Validation and test call the predictor on one scene. It returns class logits on the image grid. The metrics compare the class with the highest logit at each pixel against the reference mask.

In [ ]:
import kornia.augmentation as K
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex

class Segmenter(L.LightningModule):
    def __init__(
        self, model: ModelChain, predictor: LandCoverPredictor,
        num_classes: int, ignore_index: int,
    ) -> None:
        super().__init__()
        self.model = model
        self.predictor = predictor

        self.augment = K.AugmentationSequential(
            K.RandomHorizontalFlip(),
            K.RandomVerticalFlip(),
            data_keys=["image", "mask"],
            keepdim=True,
        )

        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(
                    num_classes=num_classes, ignore_index=ignore_index
                ),
                "accuracy": MulticlassAccuracy(
                    num_classes=num_classes, ignore_index=ignore_index, average="micro"
                ),
            },
            prefix="val_",
        )
        self.test_metrics = self.val_metrics.clone(prefix="test_")

        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction="sum")

    def on_save_checkpoint(self, checkpoint: dict) -> None:
        checkpoint["stages"] = self.model.stage_specs

    def forward(self, inputs: dict[str, torch.Tensor]) -> torch.Tensor:
        return self.model(**inputs)

    def training_step(self, batch, batch_index: int) -> torch.Tensor:
        inputs, label = batch
        inputs["image"], label = self.augment(inputs["image"], label)
        logits = self(inputs)
        valid_pixels = (label != self.criterion.ignore_index).sum()
        loss = self.criterion(logits, label) / valid_pixels.clamp_min(1)
        self.log("train_loss", loss, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch: tuple[xr.DataTree, torch.Tensor], batch_index: int) -> None:
        image, label = batch
        scene_logits = self.predictor(self.model, image)
        logits = torch.from_numpy(scene_logits.values)

        valid_pixels = (label != self.criterion.ignore_index).sum().item()
        if valid_pixels:
            loss = self.criterion(logits.unsqueeze(0), label.unsqueeze(0)) / valid_pixels
            self.log("val_loss", loss, batch_size=valid_pixels)

        prediction = logits.argmax(0).to(self.device)
        self.val_metrics.update(prediction, label.to(self.device))
        self.log_dict(self.val_metrics, batch_size=1)

    def test_step(self, batch: tuple[xr.DataTree, torch.Tensor], batch_index: int) -> None:
        image, label = batch
        scene_logits = self.predictor(self.model, image)
        prediction = torch.from_numpy(scene_logits.values).argmax(0).to(self.device)
        self.test_metrics.update(prediction, label.to(self.device))
        self.log_dict(self.test_metrics, batch_size=1)

    def configure_optimizers(self) -> torch.optim.AdamW:
        return torch.optim.AdamW(self.parameters(), lr=1e-4)


predictor = LandCoverPredictor(size=CHIP_SIZE)
module = Segmenter(model, predictor, num_classes, ignore_index)

In [ ]:
import warnings

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

warnings.filterwarnings("ignore", ".*does not have many workers.*")
logger = TensorBoardLogger("lightning_logs", name="03_inputs", version=0)
checkpoint = ModelCheckpoint(
    dirpath="data/checkpoints/03_inputs", save_last=True, save_top_k=0,
    enable_version_counter=False,
)

trainer = L.Trainer(
    max_epochs=6, devices=1,
    log_every_n_steps=1,
    logger=logger,
    callbacks=[checkpoint],
)
trainer.fit(module, datamodule=data)

## Evaluate on held-out scenes

Lightning now asks the DataModule for its test split. The model scores each scene after merging all its chips. Notebook 4 will restore the checkpoint from this run.

In [ ]:
trainer.test(module, datamodule=data)

## Predict on a validation sample

Use `read_sample(0)` as in notebook 1. Only the image goes to the predictor. Notebook 4 applies it to imagery without a reference mask.

In [ ]:
val_image, _ = data.val.read_sample(0)
sample = val_image.gs.to_tree("image")

model.eval()
scene_logits = predictor(model, sample)
scene_logits

Pick the class with the largest merged logit at each pixel. In notebook 4, the postprocessor will do this and attach the legend as part of the saved pipeline.

In [ ]:
cover = scene_logits.argmax("band").astype("uint8")
cover.gs.write_flags(class_names, class_colors).gs.plot(title="Validation prediction")